# Mine weather data

##### From https://open-meteo.com/en/docs/historical-weather-api

In [37]:
import pandas as pd

In [58]:
counties = pd.read_csv("uscounties.csv")

first_counties = [17031, 34003, 42101, 39035, 34021, 39049, 34003, 24005, 51059, 42003, 24031]
                      
important_counties = [17031, 17043, 54039, 26077, 51003, 39049, 39003, 39017, 39113, 39061, 21067, 21111, 21199, 51191, 51003, 
                      24021, 39035, 39095, 39153, 42019, 42007, 42049, 42013, 42015, 42071, 42079, 42069, 42107, 42133, 42101, 
                      42091, 42029, 34031, 34003, 34023, 34021, 34025, 34019, 34007, 34029, 24510, 24005, 24031, 24033, 11001, 
                      24045, 24025, 10003, 51059, 51153, 51810]

remaining_counties = [54039, 51003, 39003, 21199, 51191, 51003, 42007, 42013, 42015, 42107, 34019, 24045]
counties = counties[counties['county_fips'].isin(remaining_counties)]

print(counties)

         county county_ascii        county_full  county_fips state_id  \
379     Kanawha      Kanawha     Kanawha County        54039       WV   
406      Beaver       Beaver      Beaver County        42007       PA   
467  Schuylkill   Schuylkill  Schuylkill County        42107       PA   
499   Hunterdon    Hunterdon   Hunterdon County        34019       NJ   
521       Blair        Blair       Blair County        42013       PA   
544   Albemarle    Albemarle   Albemarle County        51003       VA   
589    Wicomico     Wicomico    Wicomico County        24045       MD   
601       Allen        Allen       Allen County        39003       OH   
824     Pulaski      Pulaski     Pulaski County        21199       KY   
877    Bradford     Bradford    Bradford County        42015       PA   
941  Washington   Washington  Washington County        51191       VA   

        state_name      lat      lng  population  
379  West Virginia  38.3365 -81.5281      178198  
406   Pennsylvania  4

In [65]:
import os
import pandas as pd
import requests_cache
from openmeteo_requests import Client
from retry_requests import retry
import time 


In [62]:
# ------------------------------
# Setup Open-Meteo client
# ------------------------------
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = Client(session=retry_session)


In [ ]:

output_dir = "inputs/Weather Data"
os.makedirs(output_dir, exist_ok=True)


In [67]:
# ------------------------------
# Loop through counties dataframe
# ------------------------------
for idx, row in counties.iterrows():

    lat = row["lat"]
    lng = row["lng"]
    county_name = str(row["county"])
    state_code = str(row["state_id"])    # or whatever column you have

    # clean filename: remove spaces and bad characters
    filename = f"{county_name}_{state_code}.csv"
    filename = filename.replace(" ", "_").replace("/", "-")
    filepath = os.path.join(output_dir, filename)

    # ------------------------------
    # API parameters for this county
    # ------------------------------
    params = {
        "latitude": lat,
        "longitude": lng,
        "start_date": "1993-06-01",
        "end_date":   "2025-11-15",
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m"
        ],
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
    }

    # Request data
    responses = openmeteo.weather_api(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params
    )
    response = responses[0]

    # ------------------------------
    # Parse hourly data
    # ------------------------------
    hourly = response.Hourly()

    # Must match order in params["hourly"]
    hourly_temperature = hourly.Variables(0).ValuesAsNumpy()
    hourly_humidity    = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind        = hourly.Variables(2).ValuesAsNumpy()

    # Build timestamps
    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(),     unit="s", utc=True),
            end   = pd.to_datetime(hourly.TimeEnd(),  unit="s", utc=True),
            freq  = pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly_temperature,
        "relative_humidity_2m": hourly_humidity,
        "wind_speed_10m": hourly_wind,
    }

    df = pd.DataFrame(hourly_data)

    # Save CSV
    df.to_csv(filepath, index=False)

    time.sleep(30)